# Simulating and Predicting GluSnFR Waveform Structure from Retinal Depth

This notebook is based on the attached **GluSnFR Bipolar Cell Waveform Clustering and Depth Analysis** workflow.

## Goal

Simulate GluSnFR ΔF/F waveforms from bipolar-cell-layer pixels across retinal depth, then train machine-learning models to predict waveform structure from depth.

The workflow answers:

```text
Given a pixel's retinal depth, can we predict the expected waveform class or waveform shape?
```

## Assumptions carried over from the attached notebook

- 2-photon GluSnFR imaging
- Frame rate: 50 Hz
- One ΔF/F waveform per pixel
- Known stimulus onset
- 7 retinal depth levels
- Different depths are enriched for different bipolar-cell waveform structures
- Output is a simulated dataset plus ML models that predict waveform class and waveform shape from depth

This is a **simulation notebook**, so it does not require real input CSV files.


## Library roles

| Library | Purpose in this notebook |
|---|---|
| `numpy` | simulate time axes, waveform equations, noise, and arrays |
| `pandas` | store simulated pixel metadata, features, and model outputs |
| `matplotlib` | minimal plotting of simulated and predicted waveforms |
| `scipy.stats` | summarize and compare simulated groups |
| `sklearn.model_selection` | split simulated data into train/test sets |
| `sklearn.preprocessing` | scale depth/features before ML |
| `sklearn.ensemble` | train Random Forest models for classification and waveform prediction |
| `sklearn.metrics` | evaluate classification and regression performance |
| `sklearn.decomposition` | PCA compression of full waveforms before prediction |
| `pathlib` | create output folders and save results |


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    mean_squared_error,
    r2_score,
)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## Step 1 — Define simulation settings

This creates the same basic timing structure as the attached notebook:

```text
50 Hz imaging = 20 ms per frame
```

The stimulus begins at frame 10. Each simulated waveform is a ΔF/F trace.


In [ ]:
# Output folders
project_dir = Path("glusnfr_depth_simulation_project")
output_dir = project_dir / "outputs"
table_dir = output_dir / "tables"
fig_dir = output_dir / "figures"

for folder in [table_dir, fig_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# Imaging / waveform timing
FRAME_RATE_HZ = 50
FRAME_INTERVAL_SEC = 1 / FRAME_RATE_HZ
N_TIMEPOINTS = 150

time_sec = np.arange(N_TIMEPOINTS) * FRAME_INTERVAL_SEC

STIM_ONSET_FRAME = 10
STIM_ONSET_SEC = STIM_ONSET_FRAME * FRAME_INTERVAL_SEC

# Depth structure
N_DEPTHS = 7
DEPTH_IDS = np.arange(1, N_DEPTHS + 1)
DEPTH_UM = np.linspace(0, 60, N_DEPTHS)

# Number of simulated pixels per depth
N_PIXELS_PER_DEPTH = 1200

print("Timepoints:", N_TIMEPOINTS)
print("Frame interval:", FRAME_INTERVAL_SEC, "sec")
print("Depth IDs:", DEPTH_IDS)


## Step 2 — Define waveform generator functions

Each function simulates a different kind of bipolar-cell-like GluSnFR response.

The exact waveform labels are simulation labels, not definitive cell identities.

The functions model:
- transient ON responses
- sustained ON responses
- delayed ON responses
- biphasic responses
- OFF-like negative responses
- mixed responses
- weak/noisy responses

These waveform structures are assigned different probabilities depending on depth.


In [ ]:
def alpha_response(t, onset, amplitude, tau_rise, tau_decay):
    '''
    Simulates a smooth calcium/glutamate-like transient response.

    Purpose:
    Creates a response that rises after stimulus onset and then decays.
    '''
    x = np.maximum(t - onset, 0)
    response = amplitude * (1 - np.exp(-x / tau_rise)) * np.exp(-x / tau_decay)
    return response


def sustained_response(t, onset, amplitude, tau_rise, plateau_decay):
    '''
    Simulates a slower sustained response.

    Purpose:
    Represents a response that rises and remains elevated over the stimulus window.
    '''
    x = np.maximum(t - onset, 0)
    response = amplitude * (1 - np.exp(-x / tau_rise)) * np.exp(-x / plateau_decay)
    return response


def negative_response(t, onset, amplitude, tau_rise, tau_decay):
    '''
    Simulates an OFF-like negative-going response.

    Purpose:
    Represents a decrease in GluSnFR signal after stimulus onset.
    '''
    return -alpha_response(t, onset, amplitude, tau_rise, tau_decay)


def biphasic_response(t, onset, pos_amp, neg_amp):
    '''
    Simulates a biphasic waveform with an early positive component
    followed by a delayed negative component.

    Purpose:
    Represents mixed or multiphasic response structures.
    '''
    pos = alpha_response(t, onset, pos_amp, tau_rise=0.08, tau_decay=0.8)
    neg = negative_response(t, onset + 0.45, neg_amp, tau_rise=0.10, tau_decay=0.9)
    return pos + neg


def add_noise(trace, noise_sd):
    '''
    Adds Gaussian noise to a simulated waveform.

    Purpose:
    Makes simulated waveforms more realistic and prevents perfect classification.
    '''
    return trace + rng.normal(0, noise_sd, size=trace.shape)


## Step 3 — Define depth-dependent waveform probabilities

This is the main biological assumption of the simulation.

Each depth level has a different probability distribution over waveform structures.

For example:
- superficial depths may be enriched for transient ON-like waveforms
- middle depths may be enriched for sustained or biphasic waveforms
- deeper depths may be enriched for OFF-like or delayed waveforms

You can edit this table to match your expected retinal-depth organization.


In [ ]:
WAVEFORM_CLASSES = [
    "transient_on",
    "sustained_on",
    "delayed_on",
    "biphasic",
    "off_like",
    "mixed",
    "weak_noisy",
]

# Rows = depth_id 1 through 7
# Columns = waveform classes above
depth_class_probabilities = pd.DataFrame(
    [
        [0.45, 0.20, 0.10, 0.05, 0.05, 0.10, 0.05],
        [0.30, 0.30, 0.10, 0.10, 0.05, 0.10, 0.05],
        [0.15, 0.40, 0.10, 0.15, 0.05, 0.10, 0.05],
        [0.10, 0.20, 0.20, 0.25, 0.10, 0.10, 0.05],
        [0.05, 0.10, 0.25, 0.25, 0.20, 0.10, 0.05],
        [0.05, 0.05, 0.25, 0.15, 0.35, 0.10, 0.05],
        [0.05, 0.05, 0.15, 0.10, 0.50, 0.10, 0.05],
    ],
    index=DEPTH_IDS,
    columns=WAVEFORM_CLASSES,
)

# Verify each row sums to 1
assert np.allclose(depth_class_probabilities.sum(axis=1), 1.0)

depth_class_probabilities


## Step 4 — Simulate one waveform

This function chooses waveform parameters based on the assigned waveform class.

The output is a single simulated ΔF/F trace.


In [ ]:
def simulate_waveform(class_label):
    '''
    Simulates one GluSnFR waveform.

    Parameters
    ----------
    class_label : str
        Waveform class to simulate.

    Returns
    -------
    trace : np.ndarray
        Simulated ΔF/F waveform.
    '''
    baseline = rng.normal(0, 0.015)
    noise_sd = rng.uniform(0.015, 0.05)

    if class_label == "transient_on":
        trace = alpha_response(
            time_sec,
            STIM_ONSET_SEC,
            amplitude=rng.uniform(0.35, 0.90),
            tau_rise=rng.uniform(0.04, 0.10),
            tau_decay=rng.uniform(0.35, 0.90),
        )

    elif class_label == "sustained_on":
        trace = sustained_response(
            time_sec,
            STIM_ONSET_SEC,
            amplitude=rng.uniform(0.30, 0.85),
            tau_rise=rng.uniform(0.08, 0.25),
            plateau_decay=rng.uniform(2.0, 8.0),
        )

    elif class_label == "delayed_on":
        trace = alpha_response(
            time_sec,
            STIM_ONSET_SEC + rng.uniform(0.25, 0.70),
            amplitude=rng.uniform(0.25, 0.75),
            tau_rise=rng.uniform(0.08, 0.20),
            tau_decay=rng.uniform(0.7, 1.8),
        )

    elif class_label == "biphasic":
        trace = biphasic_response(
            time_sec,
            STIM_ONSET_SEC,
            pos_amp=rng.uniform(0.25, 0.70),
            neg_amp=rng.uniform(0.15, 0.50),
        )

    elif class_label == "off_like":
        trace = negative_response(
            time_sec,
            STIM_ONSET_SEC + rng.uniform(0.05, 0.30),
            amplitude=rng.uniform(0.20, 0.70),
            tau_rise=rng.uniform(0.08, 0.18),
            tau_decay=rng.uniform(0.6, 1.8),
        )

    elif class_label == "mixed":
        on = alpha_response(
            time_sec,
            STIM_ONSET_SEC,
            amplitude=rng.uniform(0.15, 0.45),
            tau_rise=rng.uniform(0.06, 0.18),
            tau_decay=rng.uniform(0.4, 1.2),
        )
        delayed = alpha_response(
            time_sec,
            STIM_ONSET_SEC + rng.uniform(0.4, 0.9),
            amplitude=rng.uniform(0.10, 0.35),
            tau_rise=rng.uniform(0.08, 0.25),
            tau_decay=rng.uniform(0.5, 1.4),
        )
        trace = on + delayed

    elif class_label == "weak_noisy":
        trace = rng.normal(0, 0.04, size=N_TIMEPOINTS)

    else:
        raise ValueError(f"Unknown waveform class: {class_label}")

    trace = trace + baseline
    trace = add_noise(trace, noise_sd)

    return trace


## Step 5 — Simulate full depth-dependent dataset

This creates:

- `metadata_df`: one row per simulated pixel
- `waveforms_df`: one row per simulated pixel and one column per timepoint
- `simulated_df`: metadata plus waveforms

The simulated data are saved so the notebook can be used as a template for later real-data workflows.


In [ ]:
metadata_rows = []
waveform_rows = []

pixel_counter = 0

for depth_id, depth_um in zip(DEPTH_IDS, DEPTH_UM):
    probs = depth_class_probabilities.loc[depth_id].values

    for _ in range(N_PIXELS_PER_DEPTH):
        pixel_id = f"pixel_{pixel_counter:07d}"
        class_label = rng.choice(WAVEFORM_CLASSES, p=probs)
        trace = simulate_waveform(class_label)

        metadata_rows.append({
            "pixel_id": pixel_id,
            "depth_id": depth_id,
            "depth_um": depth_um,
            "true_waveform_class": class_label,
        })

        waveform_rows.append({
            "pixel_id": pixel_id,
            **{f"t{i}": trace[i] for i in range(N_TIMEPOINTS)}
        })

        pixel_counter += 1

metadata_df = pd.DataFrame(metadata_rows)
waveforms_df = pd.DataFrame(waveform_rows)

simulated_df = metadata_df.merge(waveforms_df, on="pixel_id", how="inner")
time_cols = [f"t{i}" for i in range(N_TIMEPOINTS)]

metadata_df.to_csv(table_dir / "simulated_pixel_metadata.csv", index=False)
waveforms_df.to_csv(table_dir / "simulated_pixel_waveforms.csv", index=False)
simulated_df.to_csv(table_dir / "simulated_depth_waveform_dataset.csv", index=False)

print("Simulated pixels:", len(simulated_df))
display(simulated_df.head())


## Step 6 — Minimal QC plot: mean waveform per depth

This confirms that the simulation created depth-dependent waveform structure.

Only the average waveform at each depth is plotted.


In [ ]:
plt.figure(figsize=(8, 5))

for depth_id in DEPTH_IDS:
    subset = simulated_df[simulated_df["depth_id"] == depth_id]
    mean_trace = subset[time_cols].mean(axis=0).values
    plt.plot(time_sec, mean_trace, label=f"Depth {depth_id}")

plt.axvline(STIM_ONSET_SEC, linestyle="--", color="black", label="Stimulus onset")
plt.xlabel("Time (s)")
plt.ylabel("Simulated ΔF/F")
plt.title("Mean simulated waveform by depth")
plt.legend(ncol=2)
plt.tight_layout()

out_path = fig_dir / "mean_simulated_waveform_by_depth.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)


## Step 7 — Extract waveform features

These features summarize waveform structure numerically.

Features include:
- peak ΔF/F
- minimum ΔF/F
- time to peak
- time to minimum
- area under the curve
- response polarity
- early response
- late response
- sustained index

These are useful for interpreting how waveform shape changes with depth.


In [ ]:
RESPONSE_START_FRAME = STIM_ONSET_FRAME
RESPONSE_END_FRAME = N_TIMEPOINTS
EARLY_END_FRAME = min(N_TIMEPOINTS, STIM_ONSET_FRAME + int(0.8 * FRAME_RATE_HZ))
LATE_START_FRAME = min(N_TIMEPOINTS, STIM_ONSET_FRAME + int(1.2 * FRAME_RATE_HZ))

def extract_features_from_trace(trace):
    '''
    Converts one waveform into interpretable waveform features.

    Purpose:
    Reduces a full time-series waveform into biologically interpretable values.
    '''
    baseline = np.mean(trace[:STIM_ONSET_FRAME])
    corrected = trace - baseline
    response = corrected[RESPONSE_START_FRAME:RESPONSE_END_FRAME]

    peak_idx_rel = int(np.argmax(response))
    min_idx_rel = int(np.argmin(response))

    peak_value = response[peak_idx_rel]
    min_value = response[min_idx_rel]

    peak_frame = RESPONSE_START_FRAME + peak_idx_rel
    min_frame = RESPONSE_START_FRAME + min_idx_rel

    auc = np.trapz(response, time_sec[RESPONSE_START_FRAME:RESPONSE_END_FRAME])

    early_mean = np.mean(corrected[RESPONSE_START_FRAME:EARLY_END_FRAME])
    late_mean = np.mean(corrected[LATE_START_FRAME:RESPONSE_END_FRAME])

    sustained_index = late_mean / (peak_value + 1e-8)

    if abs(peak_value) > abs(min_value):
        polarity = "positive_dominant"
    elif abs(min_value) > abs(peak_value):
        polarity = "negative_dominant"
    else:
        polarity = "balanced"

    return {
        "baseline": baseline,
        "peak_dff": peak_value,
        "min_dff": min_value,
        "time_to_peak_sec": time_sec[peak_frame] - STIM_ONSET_SEC,
        "time_to_min_sec": time_sec[min_frame] - STIM_ONSET_SEC,
        "auc": auc,
        "early_mean": early_mean,
        "late_mean": late_mean,
        "sustained_index": sustained_index,
        "polarity": polarity,
    }


feature_rows = []

for _, row in simulated_df.iterrows():
    trace = row[time_cols].values.astype(float)
    features = extract_features_from_trace(trace)
    features.update({
        "pixel_id": row["pixel_id"],
        "depth_id": row["depth_id"],
        "depth_um": row["depth_um"],
        "true_waveform_class": row["true_waveform_class"],
    })
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)
features_df.to_csv(table_dir / "simulated_waveform_features.csv", index=False)

display(features_df.head())


## Step 8 — Predict waveform class from depth

This model asks:

```text
Can retinal depth predict the likely waveform structure class?
```

Input:
- `depth_id`
- `depth_um`

Target:
- simulated waveform class

Model:
- Random Forest classifier

This is intentionally simple because the goal is to test whether depth alone contains predictive information.


In [ ]:
X = features_df[["depth_id", "depth_um"]]
y = features_df["true_waveform_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_SEED,
)

class_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

class_model.fit(X_train, y_train)

y_pred = class_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=WAVEFORM_CLASSES),
    index=[f"true_{c}" for c in WAVEFORM_CLASSES],
    columns=[f"pred_{c}" for c in WAVEFORM_CLASSES],
)

confusion_df.to_csv(table_dir / "depth_to_waveform_class_confusion_matrix.csv")
display(confusion_df)


## Step 9 — Predict waveform features from depth

This model predicts interpretable waveform properties from depth.

Input:
- depth

Targets:
- peak ΔF/F
- minimum ΔF/F
- time to peak
- time to minimum
- AUC
- early mean
- late mean
- sustained index

This gives a compact prediction of waveform structure.


In [ ]:
feature_targets = [
    "peak_dff",
    "min_dff",
    "time_to_peak_sec",
    "time_to_min_sec",
    "auc",
    "early_mean",
    "late_mean",
    "sustained_index",
]

X = features_df[["depth_id", "depth_um"]]
Y = features_df[feature_targets]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

feature_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

feature_model.fit(X_train, Y_train)

Y_pred = feature_model.predict(X_test)

feature_prediction_scores = []

for i, target in enumerate(feature_targets):
    rmse = mean_squared_error(Y_test[target], Y_pred[:, i], squared=False)
    r2 = r2_score(Y_test[target], Y_pred[:, i])
    feature_prediction_scores.append({
        "target": target,
        "rmse": rmse,
        "r2": r2,
    })

feature_prediction_scores = pd.DataFrame(feature_prediction_scores)
feature_prediction_scores.to_csv(table_dir / "depth_to_waveform_feature_prediction_scores.csv", index=False)

display(feature_prediction_scores)


## Step 10 — Predict full waveform shape from depth using PCA

Predicting every timepoint directly can be noisy.

Instead:

1. PCA compresses each full waveform into a small number of components.
2. A regression model predicts those PCA scores from depth.
3. The predicted PCA scores are converted back into a predicted waveform.

This provides a simulated model of:

```text
depth → waveform structure
```


In [ ]:
X_depth = simulated_df[["depth_id", "depth_um"]]
Y_waveforms = simulated_df[time_cols].values.astype(float)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_depth,
    Y_waveforms,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

N_PCA_COMPONENTS = 8

pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_SEED)
Y_train_pca = pca.fit_transform(Y_train)
Y_test_pca = pca.transform(Y_test)

waveform_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

waveform_model.fit(X_train, Y_train_pca)

Y_pred_pca = waveform_model.predict(X_test)
Y_pred_waveforms = pca.inverse_transform(Y_pred_pca)

waveform_rmse = mean_squared_error(Y_test, Y_pred_waveforms, squared=False)
waveform_r2 = r2_score(Y_test, Y_pred_waveforms)

pca_summary = pd.DataFrame({
    "n_pca_components": [N_PCA_COMPONENTS],
    "explained_variance_ratio_sum": [pca.explained_variance_ratio_.sum()],
    "waveform_rmse": [waveform_rmse],
    "waveform_r2": [waveform_r2],
})

pca_summary.to_csv(table_dir / "depth_to_full_waveform_prediction_summary.csv", index=False)
display(pca_summary)


## Step 11 — Generate predicted waveform at each depth

This creates the model-predicted waveform for each depth level.

These predictions are the final simulation output:

```text
depth → predicted waveform shape
```


In [ ]:
depth_prediction_grid = pd.DataFrame({
    "depth_id": DEPTH_IDS,
    "depth_um": DEPTH_UM,
})

predicted_pca = waveform_model.predict(depth_prediction_grid)
predicted_waveforms = pca.inverse_transform(predicted_pca)

predicted_waveforms_df = pd.DataFrame(
    predicted_waveforms,
    columns=time_cols,
)
predicted_waveforms_df.insert(0, "depth_um", DEPTH_UM)
predicted_waveforms_df.insert(0, "depth_id", DEPTH_IDS)

predicted_waveforms_df.to_csv(table_dir / "predicted_waveform_by_depth.csv", index=False)

display(predicted_waveforms_df.head())


## Step 12 — Minimal final plot: predicted waveform by depth

This plot shows the model-predicted waveform structure at each depth.


In [ ]:
plt.figure(figsize=(8, 5))

for _, row in predicted_waveforms_df.iterrows():
    trace = row[time_cols].values.astype(float)
    plt.plot(
        time_sec,
        trace,
        label=f"Depth {int(row['depth_id'])}"
    )

plt.axvline(STIM_ONSET_SEC, linestyle="--", color="black", label="Stimulus onset")
plt.xlabel("Time (s)")
plt.ylabel("Predicted ΔF/F")
plt.title("Predicted waveform structure from retinal depth")
plt.legend(ncol=2)
plt.tight_layout()

out_path = fig_dir / "predicted_waveform_structure_by_depth.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)


## Step 13 — Summarize class probabilities by depth

This table estimates the probability of each waveform class at each depth.

It is useful for comparing predicted waveform identity across retinal depth.


In [ ]:
class_probabilities = class_model.predict_proba(depth_prediction_grid)

class_probability_df = pd.DataFrame(
    class_probabilities,
    columns=class_model.classes_,
)
class_probability_df.insert(0, "depth_um", DEPTH_UM)
class_probability_df.insert(0, "depth_id", DEPTH_IDS)

class_probability_df.to_csv(table_dir / "predicted_waveform_class_probability_by_depth.csv", index=False)
display(class_probability_df)


## Step 14 — Interpretation

The simulation produces three kinds of predictions:

1. **Waveform class prediction**
   - Predicts whether a waveform is transient ON, sustained ON, delayed ON, biphasic, OFF-like, mixed, or weak/noisy.

2. **Waveform feature prediction**
   - Predicts interpretable waveform features such as peak ΔF/F, AUC, timing, and sustained index.

3. **Full waveform prediction**
   - Uses PCA to predict the complete waveform shape from depth.

## How to adapt this to real data

Replace the simulated dataset with the real output from your attached notebook:

```text
pixel_metadata.csv
pixel_waveforms.csv
waveform_features.csv
cluster labels
depth_id / depth_um
```

Then train the same models using:

```text
depth → cluster label
depth → waveform features
depth → PCA waveform shape
```

For real biological inference, summarize at the recording, retina, or animal level rather than treating every pixel as a fully independent biological replicate.
